## Import Python Libraries and load environment variables

In [31]:
import os
import json
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

True

## Create a helper function to beautify llm responses in our Jupyter Notebook

In [32]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [33]:
bfmsgs(result)

Here are today’s menus from the two pages you shared. Date context pulled from the pages (today is 2026-05-07).

1) Hanyang University RE12 (HY-in) page
- Breakfast (08:00–10:00, 4,000 won)
  - Doenjang Jigae (Soybean Stew) – 685 kcal
  - Seasoned Beansprouts
  - Kimchi
  - Sweet Rice Drink
  - Note: Breakfast is listed as 4,000 won for the set
- Lunch 1 (11:00–14:30, 4,000 won)
  - Spicy Grilled Chicken With Rice (725 kcal)
  - Tofu Soybean Soup
  - Seasoned Bellflower and Cucumber
  - Kimchi
- Lunch 2 (11:00–14:30, 3,500 won)
  - Seolleongtang (Oxbone Soup) – 550 kcal
  - Seasoned Bellflower and Cucumber
  - Radish Kimchi
  - Beef, Oxbone
- Lunch Noodles (11:00–14:00, 3,000 won)
  - Naengmyeon (Noodles With Cold Soup) – 520 kcal
- Dinner (16:40–18:40, 3,500 won)
  - Pork Bulgogi With Rice – 650 kcal
  - Beansprout Soup
  - Stir-fried Potato, Kimchi
- Snacks (09:00–18:40)
  - Ramen With Ricecake Balls – 2,400 won
  - Cheese Ramen – 2,400 won
  - Egg Toast – 2,100 won
  - Cheese Toast? (listed in a few lines) – 2,100 won
  - Bowl Of Rice – 700 won
  - Cheese – 400 won
- Lunch set option (11:00–13:30, 6,500 won)
  - Multigrain Rice, plus other components (listed with assorted sides and beef/pork options in the broader page)
  - Note: This is a more elaborate set, priced at 6,500 won

2) HFSPN.co (Mondayspoon page)
- Breakfast (08:00–10:00 range shown on the page; prices around 4,000 won)
  - Doenjang Jigae (Soybean Stew) is listed as a breakfast option with a 4,000 won price in the main “Breakfast” block
  - Seasoned Beansprouts
  - Kimchi
  - Sweet Rice Drink
- Lunch (11:00–14:30 range; main options around 4,000 won)
  - Spicy Grilled Chicken With Rice
  - Tofu Soybean Soup
  - Seasoned Bellflower and Cucumber
  - Kimchi
- Additional Lunch Noodle options (3,000–4,500 won range)
  - Naengmyeon (Noodles with Cold Soup)
- Dinner (16:40–18:40 range; prices around 3,500–4,000 won)
  - Pork Bulgogi With Rice
  - Beansprout Soup
  - Stir-fried Potato, Kimchi
- Snacks (multiple items with prices around 2,000–2,400 won)
  - Ramen With Ricecake Balls
  - Cheese Ramen
  - Egg Toast
  - Ramen
  - Bowl of Rice
  - Cheese
- Lunch set / Brunch options (6,500–7,500 won range)
  - Various multi-dish sets (e.g., Multigrain Rice with beef/pork components)
  - Brunch sets including combinations like pork Bulgogi, beef dishes, seafood options, etc.

Notes
- The HY-in / RE12 page and the HFSPN page both show today’s lines with times and prices, and they include a mix of Korean staples (doenjang jjigae, seolleongtang, bulgogi, naengmyeon) plus snack items (ramen, toast) and larger lunch sets.
- Prices are in Korean won (₩) as listed on the pages.
- If you’d like, I can pull exact item-by-item translations from the HFSPN page or extract today’s menu for a particular campus or dining hall more succinctly (e.g., only main meals, or only lunch items).

Would you like me to filter this to just the main meals for today, or pull menus for a specific campus location?

## Create a dummy function that returns 1500 as KRW > USD FX

In [34]:
def get_fx(currency: str) -> float:
    """Return conversion rate from KRW to the given currency (e.g., USD)."""
    # Read API key from environment variables loaded by load_dotenv()
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing EXCHANGE_RATE_API_KEY in .env")

    # Build ExchangeRate-API endpoint with KRW as base currency
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/KRW"

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; hai5016-project/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(f"Failed to fetch exchange rates: {error}") from error

    # Optional API-level error check
    if data.get("result") != "success":
        error_type = data.get("error-type", "unknown_error")
        raise RuntimeError(f"ExchangeRate API error: {error_type}")

    rates = data.get("conversion_rates", {})
    code = currency.upper()

    if code not in rates:
        raise ValueError(f"Currency code not found: {code}")

    return float(rates[code])

In [35]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [37]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    # Decode HTML bytes to text
    text = raw.decode("utf-8", errors="replace")

    # Return all page text (not only div id="wrapper")
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ", strip=True)
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [38]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [39]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_fx],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [40]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at university campuses? Find it at https://www.hanyang.ac.kr/re12 and http://hfspn.co/. And convert the KRW price into AZN."}]}
)
bfmsgs(result)

I checked the two pages you gave. Here’s what I could pull:

1) Hanyang University page (https://www.hanyang.ac.kr/re12)
- The page text I retrieved is largely a mix of site navigation and sections (HY-in, ERICA, notices, etc.) and does not present a clear, readable daily campus dining menu. I can’t reliably extract today’s food items or prices from this static text. If you have a specific dining hall page on HY-in (e.g., a “HY-in MENU” or a campus cafeteria page) I can try that, or I can try a different URL you provide.

2) HUFS Spoon / HFSPN (http://hfspn.co/)
- This page clearly lists daily menu items with KRW prices. Here are the items I see for today, grouped by meal:

- Breakfast (08:00–10:00): 4,000 won
  - Doenjang Jigae (Soybean Stew) with side dishes, Sweet Rice Drink
- Lunch 1 (11:00–14:30): 4,000 won
  - Spicy Grilled Chicken With Rice
  - Tofu Soybean Soup
  - Seasoned Bellflower and Cucumber, Kimchi
- Lunch 2 (11:00–14:30): 3,500 won
  - Seolleongtang (Oxbone Soup)
  - Seasoned Bellflower and Cucumber, Radish Kimchi
- Lunch Noodles (11:00–14:00): 3,000 won
  - Naengmyeon (Noodles with Cold Soup)
- Dinner (16:40–18:40): 3,500 won
  - Pork Bulgogi With Rice
  - Beansprout Soup, Stir Fried Potato, Kimchi
- Snack options:
  - Ramen With Ricecake Balls (2,400 won)
  - Cheese Ramen (2,400 won)
  - Toast (1,800 won)
  - Ramen (2,100 won)
  - Bowl Of Rice (700 won)
  - Cheese (400 won)

- Afternoon/Lunch specials shown in the page (example prices):
  - 6,500 won items often include: Multigrain Rice with various sides
  - 7,500 won items: Mala Rose Tteokbokki with Beef Loin, etc.
  - 4,500 won items: Spicy Cold Springy Noodles with Beef Loin, etc.
  - 6, results continue with additional dishes (pork Bulgogi, spicy beef dishes, etc.)

Conversion to AZN (approximate)
- Conversion rate used: 1 KRW ≈ 0.001173 AZN (as shown by the FX tool result: about 0.001173)
- 1,000 KRW ≈ 1.17 AZN (roughly)
- Example conversions from the HUFS Spoon menu above:
  - 4,000 won = ~4.69 AZN
  - 3,500 won = ~4.11 AZN
  - 3,000 won = ~3.52 AZN
  - 2,400 won = ~2.82 AZN
  - 2,100 won = ~2.46 AZN
  - 1,800 won = ~2.11 AZN
  - 700 won = ~0.82 AZN
  - 4,500 won = ~5.28 AZN
  - 4,500–7,500 won range (typical daily specials) = roughly 5.28–8.80 AZN

Notes and next steps
- The HY-in page from Hanyang University did not yield a readable daily menu in the fetched text. If you want, I can try a different HY-in dining page (e.g., HY-in cafeteria pages) or you can share a direct link to today’s menu.
- The HUFS Spoon (HFSPN) page provides a fairly complete today’s menu with KRW prices; I’ve included both KRW and approximate AZN conversions above.
- If you’d like, I can:
  - Pull more items from HFSPN (if you want a longer list or a full day’s menu).
  - Try to fetch alternative HY-in pages for a clearer campus dining menu.
  - Convert any additional KRW prices you encounter into AZN.